# Do protein building blocks survive Venus's acid? Check the lab's answer yourself

You'll read real NMR data and decide which molecules survive months in 98% sulfuric acid, the acid
at the base of Venus's clouds. This notebook carries three bootcamp steps, in order:

| Part | What you do | Bootcamp step | Time |
|---|---|---|---|
| 0 | Read one spectrum, then predict Ala-Ala's | How can you tell whether a molecule falls apart in acid? | 40 min |
| 1 | Judge alanine over a month | Does a single amino acid survive a month in Venus's acid? | 40 min |
| 2 | Load Ala-Ala in time order and stack it | Do protein building blocks survive Venus's acid? | 15 min |
| 3 | Compare 1 day with 4 months | the same step | 20 min |
| 4 | Record your decision | the same step | 15 min |
| 5 | Compare your decision with the published finding | the same step | 5 min |

The molecules:

* **alanine**: a single amino acid, to check the method on.
* **Ala-Ala**: two alanines joined by a peptide bond, the bond that links amino acids in every
  protein. You solve this case.
* **Gly-Gly**: two glycines joined the same way, worked through for you as the comparison.

Run the cells in order (Shift+Enter). Boxes headed **RECORD YOUR ANSWER** are where you write
something.

Data: Petkowski, M. D. Seager, Bains and S. Seager, Zenodo [10.5281/zenodo.11223995](https://doi.org/10.5281/zenodo.11223995), CC BY 4.0 (paper: *Scientific Reports* 14, 17083, 2024). Alanine and glycine: M. D. Seager, S. Seager, Bains and Petkowski, Zenodo [10.5281/zenodo.8381013](https://doi.org/10.5281/zenodo.8381013), CC BY 4.0 (paper: *Astrobiology* 24, 386, 2024). Alaninamide: Zenodo [10.5281/zenodo.11122291](https://doi.org/10.5281/zenodo.11122291) (paper: *ACS Omega* 10, 9623, 2025).

In [ ]:
# Setup. In Colab this first installs the training package and the Rakiura helper (about 30 s).
import importlib.util

if importlib.util.find_spec("nmrtrain") is None:
    %pip install -q "nmrtrain @ git+https://github.com/Rakiura-Inc/morning-star-nmr@main"
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"
import matplotlib.pyplot as plt
import nmrglue as ng  # the standard Python library for reading and processing NMR data
import numpy as np
from scipy.signal import find_peaks, peak_widths

import nmrtrain as nt  # this course's plumbing: downloads, checks, your answers, the hand-in

NOTEBOOK = "notebooks/01_capstone_ala_ala_vs_gly_gly.ipynb"
MODE = nt.portal.connect(NOTEBOOK)

> **CONNECT TO RAKIURA (optional, recommended)**
>
> **What it does:** ticks your checklist on the step as you go and attaches your figures to it. Nothing is sent until you connect.
>
> **How:** run the cell below, open the link it prints, check the code matches the one printed here, and click Allow. In Colab you do this once per session. To skip the click, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; get the key in Rakiura under Settings > Notebook key).
>
> **Skipping it is fine:** your work is kept on this computer, and you can attach your files on the step page instead.

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = nt.portal.connect(NOTEBOOK, sign_in=True)

## Part 0: Read one spectrum, then predict

### What NMR measures

In a strong magnet, hydrogen nuclei absorb radio waves at slightly different frequencies depending
on their surroundings. A signal's position depends on the hydrogen's surroundings: mostly the atoms
within two or three bonds of it. Hydrogens in the same surroundings give one signal together. A
group is a set of atoms that acts as a unit, such as a methyl (CH3). The same kind of group at a
different place in the molecule has different neighbours, so it gives its own signal. A signal's
position is its chemical shift, in parts per million (ppm): how far its frequency sits from a
reference frequency, set as 0 ppm, as a fraction of the instrument's frequency. The instrument sets
this scale when it records; these samples contain no compound that marks 0 ppm, and their spectra
show no peak there. By convention, high ppm is drawn on the left. "1H NMR" is hydrogen NMR; "13C
NMR" is carbon NMR.

Here, signal and peak mean the same thing: the response of one group of hydrogens. One signal can
show as several close lines; a line is one piece of a split signal. This is splitting, caused by
hydrogens on the next atom: a methyl next to one alpha hydrogen shows as two close lines (a
doublet), and a hydrogen next to a methyl shows as four (a quartet).

Amino acids share one backbone and differ in their side chain. Two joined by a peptide bond make a
dipeptide, such as Ala-Ala (two alanines) or Gly-Gly (two glycines). Two terms you will use often:
the methyl group (CH3) on alanine's side chain, and the alpha hydrogen, on the carbon next to the
nitrogen.

Watch MIT's short video [What is Nuclear Magnetic Resonance?](https://video.odl.mit.edu/videos/8b90da5f31784aa4b263734337ef6808/)
if you want the physics. For why signals move, see [chemical shift explained](https://chem.libretexts.org/Bookshelves/Organic_Chemistry/Book%3A_Organic_Chemistry_with_a_Biological_Emphasis_v2.0_%28Soderberg%29/05%3A_Structure_Determination_Part_II_-_Nuclear_Magnetic_Resonance_Spectroscopy/5.05%3A_The_Basis_for_Differences_in_Chemical_Shift).

### Predict: glycine

Glycine has hydrogens on carbon, nitrogen and oxygen. How many signals will its spectrum in 98% acid
show? Decide before you run the next cell.

In [ ]:
# How every spectrum in this notebook is read, with nmrglue.
def read_spectrum(zip_path):
    """The spectrum the instrument processed, with its ppm axis and labels (sample, acid, time)."""
    folder = nt.io.experiment_folder(zip_path)  # unzipped once, into nmr_data/
    # Bruker keeps the spectrum it processed in pdata/1; dic holds the instrument's settings.
    dic, data = ng.bruker.read_pdata(f"{folder}/pdata/1")
    udic = ng.bruker.guess_udic(dic, data)  # the axis settings, read from dic
    ppm = ng.fileiobase.uc_from_udic(udic).ppm_scale()  # the instrument's ppm scale, high to low
    return nt.io.labelled(zip_path, dic, ppm, data)

In [ ]:
# Glycine in 98% acid after 1 day, from the lab's amino-acid spectra on Zenodo.
AMINO_ACID_SPECTRA = "8381013"  # Zenodo record: M. D. Seager et al., CC BY 4.0
glycine_files = nt.data.fetch(AMINO_ACID_SPECTRA, "glycine", acid="98%", times=["1day"])
glycine_1day = read_spectrum(glycine_files[0])  # hydrogen (1H) spectrum, one time point

In [ ]:
def scaled(spectrum, scale_region):
    """The intensity divided by its tallest point inside scale_region (low, high ppm), so that point
    is 1. With scale_region None, the tallest point anywhere: in this acid, the acid peak."""
    if scale_region is None:
        return spectrum.intensity / spectrum.intensity.max()
    _, inside = spectrum.region(*scale_region)
    return spectrum.intensity / inside.max()

In [ ]:
def plot_stack(spectra, ppm_range, scale_region, title, labels=None, ax=None, offset=1.2):
    """Traces stacked bottom to top in the order given, each scaled by scaled(), then raised."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 5))
    colours = plt.cm.viridis(np.linspace(0.0, 0.85, len(spectra)))  # dark to light, short of yellow
    for number, spectrum in enumerate(spectra):
        label = labels[number] if labels else spectrum.time_label
        trace = scaled(spectrum, scale_region) + offset * number
        ax.plot(spectrum.ppm, trace, lw=0.8, color=colours[number], label=label)
    ax.set_xlim(ppm_range[1], ppm_range[0])  # NMR convention: high ppm on the left
    ax.set_ylim(-0.2, offset * (len(spectra) - 1) + 1.4)
    ax.set_yticks([])  # the heights are scaled, so their numbers mean nothing
    ax.set_xlabel("chemical shift (ppm)")
    ax.set_title(title)
    handles, names = ax.get_legend_handles_labels()  # listed top trace first, like the stack
    ax.legend(handles[::-1], names[::-1], frameon=False, fontsize=8, loc="upper left")
    return ax.figure

In [ ]:
# scale_region=None scales the trace to its tallest point, so the whole spectrum fits,
# acid peak and all.
title = "glycine in 98% acid, 1 day"
fig_glycine = plot_stack([glycine_1day], ppm_range=(0, 13), scale_region=None, title=title)
plt.show()

In [ ]:
def group_lines(lines, group_ppm):
    """Lines (ppm, height), low ppm first, closer than group_ppm to the line before belong to the
    same signal: a methyl's two lines are 0.016 ppm apart."""
    groups = []
    previous_ppm = None
    for ppm, height in lines:
        if previous_ppm is not None and ppm - previous_ppm < group_ppm:
            groups[-1].append((ppm, height))
        else:
            groups.append([(ppm, height)])
        previous_ppm = ppm
    return groups

In [ ]:
def signal_from(lines):
    """One signal from its lines: their height-weighted centre, how many, and the tallest."""
    total_height = sum(height for _, height in lines)
    centre_ppm = sum(ppm * height for ppm, height in lines) / total_height
    tallest = max(height for _, height in lines)
    return {
        "ppm": round(float(centre_ppm), 3),
        "lines": len(lines),
        "height": round(float(tallest), 3),
    }

In [ ]:
def pick_peaks(spectrum, ppm_range, min_height, group_ppm=0.03):
    """The signals between two ppm values, high ppm first, each {"ppm", "lines", "height"}.
    min_height and the heights are shares of the tallest point in ppm_range."""
    ppm, intensity = spectrum.region(*ppm_range)
    heights = intensity / intensity.max()
    # A line must reach min_height and stand 0.02 above the dips beside it (ripples do not).
    found, _ = find_peaks(heights, height=min_height, prominence=0.02)
    lines = sorted(zip(ppm[found], heights[found]))  # (ppm, height), low ppm first
    signals = [signal_from(group) for group in group_lines(lines, group_ppm)]
    return signals[::-1]  # high ppm first, the way a spectrum is read

In [ ]:
# The peaks the picker finds between 3 and 13 ppm: the three signals the next box names.
glycine_peaks = pick_peaks(glycine_1day, ppm_range=(3.0, 13.0), min_height=0.02)
for peak in glycine_peaks:
    print(f"glycine peak at {peak['ppm']:.2f} ppm")

### Three signals

- The acid's own peak near 11.5 ppm, the tallest peak (the last part of this section says why).
  Find it first: it is your landmark. Near 11.5 ppm, on the left, it shows the axis runs the right
  way; with the axis mirrored it lands near 0 to 0.6 ppm instead.
- The NH3+ hydrogens near 6.8 ppm (the amino group, which carries an extra hydrogen in acid).
- The CH2 hydrogens on carbon near 4.4 ppm.

The hydrogen on the carboxyl group (COOH) does not show as its own peak: it swaps with the acid so
fast that it merges into the acid's peak.

### Same molecule, different liquid

Heavy water (D2O) is water made with deuterium, a heavy form of hydrogen. In acid, Gly-Gly's two CH2
signals (its hydrogens on carbon) sit about 1 ppm further left than in heavy water. Most reference
tables and databases list ppm values measured in water, so do not use them for these spectra.

### Some peaks fade without breakdown

The acid in these samples is deuterated sulfuric acid (D2SO4) in heavy water: at least 99.5% of its
hydrogen is deuterium, which gives no signal in hydrogen NMR. The acid's peak near 11.5 ppm comes
from the small amount of ordinary hydrogen left in it, plus the hydrogens the sample's NH3+ and COOH
groups hand over to the acid when they swap with its deuterium. The acid far outnumbers the sample,
so even that small share adds up to about as many hydrogens as the sample has, and all of them fall
in one narrow peak. That makes it the tallest peak. Because it collects the hydrogens the sample
hands over, its size can change over time.

Hydrogens on nitrogen or oxygen swap with the acid's deuterium, at different speeds. The one on the
carboxyl group swaps so fast that it never shows as its own peak. The NH3+ hydrogens swap slowly, so
their peak shrinks over days to weeks. Neither is breakdown. Hydrogens on carbon do not swap this
way, or swap far more slowly, so they are the ones to track.

**In short:** find the acid peak first, then watch only the hydrogens on carbon.

### Now predict Ala-Ala and Gly-Gly

In this acid the amine end of the chain carries an extra hydrogen (H3N+). Hydrogens on nitrogen or oxygen
fade (see Some peaks fade without breakdown, above), so count only the hydrogens on carbon.

### What you need

Every group of hydrogens on carbon gives a signal, and two groups share one only if their
surroundings are identical. The drawing below numbers each group; decide which sit in identical
surroundings.

| Hydrogens on carbon | Typical position in 98% acid |
|---|---|
| methyl group (CH3) | near 2 ppm |
| alpha hydrogen (on the carbon next to the nitrogen) | near 5 ppm |
| glycine CH2 in a dipeptide | near 4.6 ppm |

In water these signals sit about 1 ppm lower (see Same molecule, different liquid, above).

In [ ]:
# Every group of hydrogens on carbon, circled and numbered.
fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
ala_ala_groups = {"S1": 1, "A1": 2, "A2": 3, "S2": 4}  # S: side-chain methyl, A: alpha carbon
gly_gly_groups = {"A1": 1, "A2": 2}
nt.plot.draw_dipeptide(
    axes[0], "Ala", "Ala-Ala: groups of hydrogens on carbon", circle=ala_ala_groups
)
nt.plot.draw_dipeptide(
    axes[1], "Gly", "Gly-Gly: groups of hydrogens on carbon", circle=gly_gly_groups
)
plt.show()

> **RECORD YOUR ANSWER: predict the two spectra before you see them**
>
> **What gets saved:** `MY_PREDICTION`, seven short answers, saved with the time so your hand-in shows you predicted first.
>
> **Where to look:** the circled drawings and the table of typical positions above.
>
> **How to fill this in:**
> - `AA_signals`: how many signals Ala-Ala's hydrogens on carbon give, a whole number like `3`
> - `AA_methyl_ppm`: roughly where its methyl signals sit, a number in ppm like `2.0`
> - `AA_alpha_ppm`: roughly where its alpha hydrogen signals sit, a number in ppm
> - `GG_signals`: how many signals Gly-Gly's hydrogens on carbon give
> - `GG_ch2_ppm`: roughly where its CH2 signals sit, a number in ppm
> - `breaks_down`: which breaks down within four months: `"Ala-Ala"`, `"Gly-Gly"`, `"both"`, `"neither"` or `"unsure"`
> - `fading`: which hydrogens on nitrogen or oxygen will shrink or not show, and why, a few words in quotes
> - Leave `...` for any you skip.

<details><summary>Stuck? Open hint 1</summary>

Look at where each circled group sits along the chain. Are the surroundings of group 1 the
same as those of group 4?

</details>

<details><summary>Stuck? Open hint 2</summary>

The two alanines are not in identical places: one is at the start of the chain (next to
H3N+), the other at the end (next to the carboxyl group, COOH). So each gives its own methyl signal and
its own alpha hydrogen signal. The three hydrogens within one CH3 are identical and give one signal
together. The two CH2 groups of Gly-Gly differ in the same way.

</details>

Your prediction is saved with the time. For the bootcamp step, paste these seven answers into the
hand-in box (or attach a sketch), with one thing you did not understand.

In [ ]:
MY_PREDICTION = {
    "AA_signals": ...,  # a whole number: how many signals do Ala-Ala's hydrogens on carbon give?
    "AA_methyl_ppm": ...,  # a number in ppm: roughly where are its methyl (CH3) signals?
    "AA_alpha_ppm": ...,  # a number in ppm: its alpha hydrogens (on the carbon by the nitrogen)?
    "GG_signals": ...,  # a whole number: how many signals do Gly-Gly's hydrogens on carbon give?
    "GG_ch2_ppm": ...,  # a number in ppm: roughly where are its CH2 signals?
    "breaks_down": ...,  # "Ala-Ala", "Gly-Gly", "both", "neither" or "unsure"
    "fading": ...,  # in words: which hydrogens on N or O shrink or vanish, and why
}
nt.record.predict("capstone", MY_PREDICTION)

In [ ]:
# Reveal: runs only after you predicted. It checks your reasoning, not this dataset.
if nt.record.revealed("capstone"):
    key = nt.answers.load("capstone", quiet=True)
    prediction_feedback = nt.checks.check_prediction(MY_PREDICTION, key["prediction"])
    nt.checks.show(prediction_feedback, "Your predictions")

In [ ]:
# The answer, drawn: each signal lettered, coloured by kind of group.
if nt.record.has_prediction("capstone"):  # if not, the cell above said what to do
    methyl, alpha, ch2 = nt.plot.PALE_ORANGE, nt.plot.PALE_BLUE, nt.plot.PALE_GREEN
    ala_ala_letters = {"S1": ("a", methyl), "S2": ("b", methyl)}  # the methyls
    ala_ala_letters.update({"A1": ("c", alpha), "A2": ("d", alpha)})  # the alpha hydrogens
    gly_gly_letters = {"A1": ("a", ch2), "A2": ("b", ch2)}
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
    nt.plot.draw_dipeptide(
        axes[0], "Ala", "Ala-Ala: 4 signals from hydrogens on carbon", ala_ala_letters
    )
    nt.plot.draw_dipeptide(
        axes[1], "Gly", "Gly-Gly: 2 signals from hydrogens on carbon", gly_gly_letters
    )
    plt.show()

In [ ]:
if nt.record.has_prediction("capstone"):
    print(
        "Each of the two alanines gives its own methyl and alpha hydrogen signal, "
        "because the two are not in"
    )
    print(
        "identical places: one is at the start of the chain, the other at the end. "
        "Gly-Gly's two CH2 groups differ the same way."
    )
    print("Next: try the method on alanine (Part 1), then load Ala-Ala's spectra (Part 2).")
    print("You write down where each signal really sits in Part 3.")
    nt.portal.done("capstone-predict")

## Part 1: Judge alanine over a month

Before judging a peptide, check the method on a single amino acid. The next cell downloads alanine
in 98% acid at 1 day, 8 days and 1 month and stacks it twice: the whole spectrum, and 1 to 7 ppm.
The middle file sits in the dataset's "1 week" folder; its file name gives the real time point, 8
days.

### What to look at

Alanine has two signals from hydrogens on carbon: the methyl near 2.0 ppm and the alpha hydrogen
near 4.6 ppm. Alanine's alpha hydrogen sits lower than Ala-Ala's, which are near 5 ppm. The NH3+
hydrogens sit near 6.8 ppm.

Look at the peaks from hydrogens on carbon. Does any of them disappear, move or shrink, and does
anything new appear? Each file's intensity is scaled differently, so compare sizes only within one
spectrum: alanine's methyl peak against its alpha hydrogen peak. Do not use the acid peak as the
yardstick here (see Part 0): it grows over the month, and every other peak shrinks against it
whether or not alanine changes.

Two things will change that do not mean alanine is breaking down. Don't count these:
- The NH3+ peak near 6.8 ppm shrinks (see Some peaks fade without breakdown, in Part 0).
- The alpha hydrogen peak near 4.6 ppm may change shape. While the NH3+ group next to it carries
  hydrogen, it splits that peak into extra lines; as those hydrogens become deuterium, the splitting
  fades. The peak stays at the same position.

**Predict:** will alanine's signals change over a month in the acid? Decide before you run the next
cell.

In [ ]:
# Alanine in 98% acid at 1 day, 8 days (the "1 week" folder) and 1 month.
alanine_files = nt.data.fetch(AMINO_ACID_SPECTRA, "alanine", acid="98%")
alanine = [read_spectrum(path) for path in alanine_files]
for spectrum in alanine:
    print(nt.checks.check_axis(spectrum))

In [ ]:
# Whole spectrum: each trace scaled to its tallest point, which is the acid peak.
title = "alanine in 98% acid, whole spectrum"
fig_alanine_whole = plot_stack(alanine, ppm_range=(0, 13), scale_region=None, title=title)
plt.show()

In [ ]:
# 1 to 7 ppm: each trace scaled to its tallest peak in that range, so alanine's own peaks
# fill it.
title = "alanine in 98% acid, 1 to 7 ppm"
fig_alanine_zoom = plot_stack(alanine, ppm_range=(1.0, 7.0), scale_region=(1.0, 7.0), title=title)
plt.show()

In [ ]:
# Where the picker finds peaks between 1 and 7 ppm, at each time point.
alanine_peaks_ppm = {}
for spectrum in alanine:
    peaks = pick_peaks(spectrum, ppm_range=(1.0, 7.0), min_height=0.03)
    positions = [round(peak["ppm"], 2) for peak in peaks]
    alanine_peaks_ppm[spectrum.time_label] = positions
    print(f"{spectrum.time_label:9s}", positions)

**Stuck?** If the acid peak sits near 0 to 0.6 ppm instead of 11.5, the spectrum is mirrored: its
ppm scale runs the wrong way. If that happens, or the files will not load, post a screenshot in the
lab's Discord channel.

> **RECORD YOUR ANSWER: your call on alanine**
>
> **What gets saved:** `ALA_CALL`, `ALA_PEAKS_PPM` and `ALA_NOTE`.
>
> **Where to look:** the two stacks and the peak lists above.
>
> **How to fill this in:**
> - `ALA_CALL`: your call, one of:
>   - `"survives"`: the peaks from hydrogens on carbon (near 2.0 and 4.6 ppm) are still there, in the same places, and keep their sizes relative to each other. No new peaks appear.
>   - `"side chain changed"`: the main peaks stay, but one group of peaks shifts or new ones appear next to them. The core of the molecule (its backbone) is intact, but one part of it (its side chain) reacted.
>   - `"breaks down"`: peaks from hydrogens on carbon shrink or vanish while peaks from the products appear and grow.
>   - `"unsure"`: the data do not settle it (say why in `ALA_NOTE`)
> - `ALA_PEAKS_PPM`: the peaks your call rests on, as numbers like `[1.23, 4.56]`
> - `ALA_NOTE`: anything your reviewer should know, or `""`
> - Keep the positions: in Part 4, free alanine is one of the two candidates for what Ala-Ala becomes.

<details><summary>Stuck? Open hint 1</summary>

Compare the methyl with the alpha hydrogen in each spectrum, not either of them with the acid peak.

</details>

<details><summary>Stuck? Open hint 2</summary>

A peak above 6.5 ppm that fades is hydrogen on nitrogen. A peak that changes shape but stays at the
same position has not moved.

</details>

In [ ]:
ALA_CALL = ""  # your call: "survives", "side chain changed", "breaks down" or "unsure"
ALA_PEAKS_PPM = []  # the peaks your call rests on, like [1.23, 4.56]
ALA_NOTE = ""  # anything your reviewer should know, or ""

nt.record.write_down("ALA_CALL", ALA_CALL, "alanine", "call")
nt.record.saved("ALA_PEAKS_PPM", ALA_PEAKS_PPM, "the peaks your call on alanine rests on")
nt.record.write_down("ALA_NOTE", ALA_NOTE, "alanine", "note")

> **SAVE YOUR HAND-IN: your alanine hand-in**
>
> **What gets saved:** in `handin/`: the whole stack (the acid peak is the tallest, near 11.5 ppm), the 1 to 7 ppm stack, and `verdict.json` with your call, its peaks and the peak lists printed above.
>
> **How to fill this in:**
> - Run the cell. These files are your hand-in for the bootcamp step "Does a single amino acid survive a month in Venus's acid?". The printed peak lists count as the labels on your pictures.
> - Nothing is sent for review until you run `nt.portal.submit()` in the last line of the cell.

<details><summary>Stuck? Open hint 1</summary>

If a figure is missing from the list it prints, run the cell that draws it, then this cell again.

</details>

<details><summary>Stuck? Open hint 2</summary>

If it says a name is not defined, run the notebook from the top (Runtime > Run all in Colab).

</details>

In [ ]:
record = nt.record.handin(
    "alanine-verdict",
    figures={"stack_alanine_whole": fig_alanine_whole, "stack_alanine_1to7": fig_alanine_zoom},
    verdict={
        "call": ALA_CALL,
        "peaks_ppm": ALA_PEAKS_PPM,
        "note": ALA_NOTE,
        "peak_lists_ppm": alanine_peaks_ppm,
    },
    numbers={"ALA_call": ALA_CALL},
)
# nt.portal.submit()   # uncomment to send this for review

## Part 2: Load Ala-Ala and look

Each spectrum is one zip file from a 400 MHz spectrometer, already processed by the instrument.
The next cell downloads the ten you need (about 5 MB) and lists them.

In [ ]:
from pathlib import Path

DIPEPTIDE_SPECTRA = "11223995"  # Zenodo record: Petkowski et al., CC BY 4.0
ala_ala_files = nt.data.fetch(DIPEPTIDE_SPECTRA, "Ala-Ala", acid="98%")
by_time = {nt.data.time_of(path): path for path in ala_ala_files}  # time label -> file
print("\nSorted by file name:")
for path in sorted(ala_ala_files):
    print("   ", Path(path).name)

### Trap: loading order

Sorting the labels as text does not put them in time order: `4Month` lands before `5week` and `8day`.

### What you need

The five time labels are `"1day"`, `"3day"`, `"8day"`, `"5week"` (5 weeks) and `"4Month"`
(4 months).

> **RECORD YOUR ANSWER: put Ala-Ala's spectra in time order**
>
> **What gets saved:** `AA_ORDER`, the five time labels as a list, first to last.
>
> **Where to look:** the file names printed above.
>
> **How to fill this in:**
> - Replace `sorted(by_time)` with a list in square brackets, each label in quotes, in time order, like `["1day", "3day", ...]` with all five.

<details><summary>Stuck? Open hint 1</summary>

Put each label into days, then sort the numbers.

</details>

<details><summary>Stuck? Open hint 2</summary>

1 day, 3 days, 8 days, 5 weeks (35 days), 4 months (about 120 days).

</details>

In [ ]:
AA_ORDER = sorted(by_time)  # change this: the five labels in time order, like ["1day", "3day", ...]
nt.record.saved("AA_ORDER", AA_ORDER, "the order the Ala-Ala spectra are stacked in, first to last")

In [ ]:
ala_ala = [read_spectrum(by_time[time]) for time in AA_ORDER]  # in the order you chose
print(nt.checks.check_order(ala_ala))
for spectrum in ala_ala:
    recorded = f"{spectrum.acquired:%Y-%m-%d}"
    print(f"   {spectrum.time_label:9s} recorded {recorded}   folder {spectrum.folder}")

Some folder names carry the wrong date: go by the time label and the recording date in the file.

### Trap: a mirrored axis

If the acid peak sits between 0 and 0.6 ppm instead of near **11.5 ppm** (see Part 0), the
spectrum is mirrored and every reading is wrong, with no error message. This matters when you process raw data (notebook 04).

In [ ]:
for spectrum in ala_ala:
    print(nt.checks.check_axis(spectrum))
print(nt.checks.check_drift(ala_ala))

Over four months the acid peak moves about 0.06 ppm higher while the peptides' own peaks stay put.
Keep the instrument's ppm scale; do not shift spectra to line up the acid peak (re-referencing).

### The stack: Gly-Gly, worked

The next cell stacks Gly-Gly's five time points, oldest at the bottom, scaled within 1 to 6.5 ppm.

In [ ]:
gly_gly_files = nt.data.fetch(DIPEPTIDE_SPECTRA, "Gly-Gly", acid="98%")
gly_gly = [read_spectrum(path) for path in gly_gly_files]  # first time point first
title = "Gly-Gly in 98% acid, 1H"
fig_gly_gly = plot_stack(gly_gly, ppm_range=(0, 13), scale_region=(1.0, 6.5), title=title)
plt.show()

### What to look at in Gly-Gly

Gly-Gly's two peaks stay in place. Gly-Gly survives at least 4 months in 98% acid (Table 2 of the
paper). One of its two CH2 peaks, near 4.77 ppm, still weakens: by 4 months it is about a third of
its day 1 size against the acid peak, while the 4.61 ppm peak keeps most of its size and no sizable
new peaks appear. The paper does not explain this, so do not count it as breakdown. From 5 weeks,
the same peak shows a small second line beside it; by 4 months the two lines are equally tall. The
NMR project has an Extra on it.

Two other changes do not mean Gly-Gly is breaking down. Don't count these:
- Peaks that get sharper after day 1. This is most likely deuterium exchange, as with alanine's
  alpha hydrogen in Part 1.
- The acid peak moving a few hundredths of a ppm: leave the ppm scale as it is (see above, under
  the drift check).

### What you need

`NORM` is the ppm region each trace is scaled to: the tallest point inside it becomes the same height
in every trace. The acid peak is far taller than the sample's peaks, so scaling to the whole spectrum
(`None`) makes them tiny and hides small changes.

> **RECORD YOUR ANSWER: choose the region to scale Ala-Ala's stack**
>
> **What gets saved:** `NORM`, two numbers in ppm in round brackets, low then high.
>
> **Where to look:** the Gly-Gly stack above, and the table of typical positions in Part 0.
>
> **How to fill this in:**
> - Run the cell once with `NORM = None` and read the checks it prints.
> - Then set `NORM` to the region you will read, like `(1.0, 6.5)`, and run it again.

<details><summary>Stuck? Open hint 1</summary>

Which ppm range holds all the hydrogens on carbon, and leaves out the acid peak?

</details>

<details><summary>Stuck? Open hint 2</summary>

Use `(1.0, 6.5)`, the region the Gly-Gly stack uses: it holds the methyls near 2 ppm and the
alpha hydrogens near 5 ppm, and leaves out the acid peak near 11.5 ppm.

</details>

In [ ]:
NORM = None  # change this: the ppm region to scale within, low then high, like (1.0, 6.5)
nt.record.saved("NORM", NORM, "the region each trace is scaled to")

title = "Ala-Ala in 98% acid, 1H"
fig_ala_ala = plot_stack(ala_ala, ppm_range=(0, 13), scale_region=NORM, title=title)
plt.show()
traps_ala_ala = nt.checks.run_traps(ala_ala, norm=NORM)
nt.portal.done("capstone-stack", figures=[fig_ala_ala])

## Part 3: Compare 1 day and 4 months

The next cell overlays each peptide's first and last spectrum from 1 to 6.5 ppm, where all the
hydrogens on carbon are. A peak picker (code that finds peaks) marks each peak; `MIN_HEIGHT` is the
smallest it marks, as a fraction of the tallest.

<details><summary>Stuck? Open hint 1</summary>

`MIN_HEIGHT = 0.10` hides any peak smaller than a tenth of the tallest. Try it, see which marks
disappear, then set it back to `0.03`.

</details>

In [ ]:
MIN_HEIGHT = 0.03  # try 0.10: which peaks would you miss?
READ_PPM = (1.0, 6.5)  # all the hydrogens on carbon of both peptides sit in here

# The 1 day and 4 month spectra, by their time labels, whatever order you stacked them in.
ala_ala_by_time = {spectrum.time: spectrum for spectrum in ala_ala}
gly_gly_by_time = {spectrum.time: spectrum for spectrum in gly_gly}
ala_ala_1day, ala_ala_4months = ala_ala_by_time["1day"], ala_ala_by_time["4Month"]
gly_gly_1day, gly_gly_4months = gly_gly_by_time["1day"], gly_gly_by_time["4Month"]

ala_ala_peaks_1day = pick_peaks(ala_ala_1day, READ_PPM, min_height=MIN_HEIGHT)
ala_ala_peaks_4months = pick_peaks(ala_ala_4months, READ_PPM, min_height=MIN_HEIGHT)
gly_gly_peaks_1day = pick_peaks(gly_gly_1day, READ_PPM, min_height=MIN_HEIGHT)
gly_gly_peaks_4months = pick_peaks(gly_gly_4months, READ_PPM, min_height=MIN_HEIGHT)

In [ ]:
def compare_peaks(first_ppm, last_ppm, tolerance_ppm=0.05):
    """Which peaks are at both times (kept), at the first only (lost) and at the last only
    (new). Two peaks within tolerance_ppm of each other count as the same peak."""
    changes = {"kept": [], "lost": [], "new": []}
    for ppm in first_ppm:
        if any(abs(ppm - later) <= tolerance_ppm for later in last_ppm):
            changes["kept"].append(ppm)
        else:
            changes["lost"].append(ppm)
    for ppm in last_ppm:
        if not any(abs(ppm - earlier) <= tolerance_ppm for earlier in first_ppm):
            changes["new"].append(ppm)
    return changes

In [ ]:
ala_ala_ppm_1day = [peak["ppm"] for peak in ala_ala_peaks_1day]
ala_ala_ppm_4months = [peak["ppm"] for peak in ala_ala_peaks_4months]
gly_gly_ppm_1day = [peak["ppm"] for peak in gly_gly_peaks_1day]
gly_gly_ppm_4months = [peak["ppm"] for peak in gly_gly_peaks_4months]
ala_ala_changes = compare_peaks(ala_ala_ppm_1day, ala_ala_ppm_4months)
gly_gly_changes = compare_peaks(gly_gly_ppm_1day, gly_gly_ppm_4months)

In [ ]:
COMPARE_OFFSET = 1.15  # how far the later trace sits above the first


def mark_peaks(ax, peaks, baseline):
    """Write each peak's position above it. baseline: the height its trace is raised by."""
    for peak in peaks:
        top = (peak["ppm"], peak["height"] + baseline)
        label = f"{peak['ppm']:.2f}"
        ax.annotate(
            label,
            top,
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            fontsize=7.5,
            rotation=90,
        )

In [ ]:
def plot_first_vs_last(ax, first, last, first_peaks, last_peaks):
    """The first spectrum (bottom) and the last (top), scaled within READ_PPM, peaks labelled."""
    title = f"{first.sample_name}: {first.time_label} (bottom) vs {last.time_label} (top)"
    plot_stack(
        [first, last], READ_PPM, scale_region=READ_PPM, title=title, ax=ax, offset=COMPARE_OFFSET
    )
    mark_peaks(ax, first_peaks, baseline=0.0)
    mark_peaks(ax, last_peaks, baseline=COMPARE_OFFSET)
    ax.set_ylim(-0.1, COMPARE_OFFSET + 1.6)  # room for the labels over the top trace

In [ ]:
fig_zoom, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_first_vs_last(
    axes[0], ala_ala_1day, ala_ala_4months, ala_ala_peaks_1day, ala_ala_peaks_4months
)
plot_first_vs_last(
    axes[1], gly_gly_1day, gly_gly_4months, gly_gly_peaks_1day, gly_gly_peaks_4months
)
plt.show()

In [ ]:
def listed(positions_ppm):
    """Positions as '1.99, 2.08', or 'none'."""
    return ", ".join(f"{ppm:.2f}" for ppm in positions_ppm) or "none"

In [ ]:
for name, changes in (("Ala-Ala", ala_ala_changes), ("Gly-Gly", gly_gly_changes)):
    kept, lost, new = listed(changes["kept"]), listed(changes["lost"]), listed(changes["new"])
    print(f"{name}: at both times {kept} | at 1 day only {lost} | at 4 months only {new}")

### What you need

Read the 1 day trace (the lower one); the number over each peak is its position in ppm. Each methyl
shows as two close lines, split by the alpha hydrogen next to it (Part 0), and the peak
picker reports them as one peak at their centre.

> **RECORD YOUR ANSWER: where do the signals really sit?**
>
> **What gets saved:** `MY_READING`, three positions in ppm, read off the 1 day traces.
>
> **Where to look:** the lower trace in each plot above and the numbers over its peaks.
>
> **How to fill this in:**
> - `AA_methyl_ppm`: one of Ala-Ala's methyl peaks, a number like `2.0`
> - `AA_alpha_ppm`: one of Ala-Ala's alpha hydrogen peaks
> - `GG_ch2_ppm`: one of Gly-Gly's CH2 peaks

<details><summary>Stuck? Open hint 1</summary>

Which of Ala-Ala's peaks sit near 2 ppm, and which near 5 ppm?

</details>

<details><summary>Stuck? Open hint 2</summary>

Ala-Ala has two methyl peaks near 2 ppm and two alpha hydrogen peaks near 5 ppm; write one of
each. Gly-Gly's two CH2 peaks sit between 4.5 and 5 ppm.

</details>

In [ ]:
MY_READING = {
    "AA_methyl_ppm": ...,  # a number in ppm, read off the Ala-Ala 1 day trace
    "AA_alpha_ppm": ...,  # a number in ppm
    "GG_ch2_ppm": ...,  # a number in ppm, read off the Gly-Gly 1 day trace
}
reading_key = nt.answers.load("capstone", quiet=True)["reading"]
reading_feedback = nt.checks.check_reading(MY_READING, reading_key, MY_PREDICTION)
nt.checks.show(reading_feedback, "Where the signals sit at 1 day")

Intact Ala-Ala has **two** methyl signals (1.99 and 2.08 ppm) and **two** alpha
hydrogen signals (4.92 and 5.17 ppm), one of each per alanine. Which are left at 4 months? A peak in
the same place may belong to a new molecule.

### Trap: peaks that sharpen after day 1

Both peptides' peaks are broader at 1 day (why: What to look at in Gly-Gly, in Part 2). The positions do not move, so it is not a
new molecule.

In [ ]:
def peak_width_hz(spectrum, near_ppm, half_window_ppm=0.06):
    """The tallest point within near_ppm +/- half_window_ppm: its position in ppm and its
    full width at half height in Hz."""
    inside = (spectrum.ppm > near_ppm - half_window_ppm) & (
        spectrum.ppm < near_ppm + half_window_ppm
    )
    ppm, intensity = spectrum.ppm[inside], spectrum.intensity[inside]
    top = int(np.argmax(intensity))
    width_points = peak_widths(intensity, [top], rel_height=0.5)[0][0]  # scipy counts in points
    ppm_per_point = abs(ppm[1] - ppm[0])
    width_hz = (
        width_points * ppm_per_point * spectrum.spectrometer_mhz
    )  # 1 ppm is 400 Hz at 400 MHz
    return float(ppm[top]), float(width_hz)

In [ ]:
for spectrum in gly_gly:
    peak_ppm, width_hz = peak_width_hz(spectrum, near_ppm=4.62)
    print(f"Gly-Gly {spectrum.time_label:9s} peak at {peak_ppm:.3f} ppm, {width_hz:4.1f} Hz wide")
nt.portal.done("capstone-compare", figures=[fig_zoom])  # ticks Part 3's comparison on Rakiura

### How much is left

Peak area is proportional to the number of hydrogens. These samples have no internal standard, so the
next cell divides each peak by the acid peak in the same file, then by that ratio at day 1:

- Under one third: little or none is left. This is clear loss.
- One third or more: no clear loss by this measure. Decide from the spectra: whether new peaks
  appear, and whether the molecule's peaks keep their sizes relative to each other.

The acid peak is not a fixed yardstick (see Part 0 and Part 1), so the cell also measures each
molecule's peaks against each other.

In [ ]:
ACID_PEAK_PPM = (10.0, 13.0)  # the acid's own peak sits between 11.4 and 12.1 ppm in these data


def peak_area(spectrum, ppm_range):
    """The sum of the intensity between two ppm values: a rough area, enough to compare peaks
    within one file."""
    _, intensity = spectrum.region(*ppm_range)
    return float(intensity.sum())

In [ ]:
def area_ratios(spectra, peak_ppm, reference_ppm):
    """At each time point, the peak's area divided by a reference peak's area in the same file."""
    ratios = []
    for spectrum in spectra:
        ratio = peak_area(spectrum, peak_ppm) / peak_area(spectrum, reference_ppm)
        ratios.append(max(ratio, 0.0))  # where a peak has gone, noise can sum below zero
    return ratios


def share_of_first(ratios):
    """Each ratio as a fraction of the first one: 1.0 at the first time point."""
    return [ratio / ratios[0] for ratio in ratios]

In [ ]:
CLEAR_LOSS_BELOW = 1 / 3  # under a third of day 1 is clear loss (see How much is left)


def how_much_is_left(fraction):
    """The two levels the steps use, from the fraction left at the last time point."""
    if fraction < CLEAR_LOSS_BELOW:
        return "little or none"
    return "no clear loss"

In [ ]:
# Each peak divided by the acid peak in the same file, as a percentage of its day 1 value.
ala_ala_by_date = nt.io.by_recording_date(ala_ala)  # your stack order may differ
gly_gly_by_date = nt.io.by_recording_date(gly_gly)
for label, spectra, peak_ppm in (
    ("Ala-Ala methyl at 1.99 ppm", ala_ala_by_date, (1.94, 2.045)),
    ("Gly-Gly CH2 at 4.61 ppm   ", gly_gly_by_date, (4.55, 4.68)),
    ("Gly-Gly CH2 at 4.77 ppm   ", gly_gly_by_date, (4.68, 4.86)),
):
    left = share_of_first(area_ratios(spectra, peak_ppm, ACID_PEAK_PPM))
    percentages = ", ".join(
        f"{spectrum.time_label} {share:.0%}" for spectrum, share in zip(spectra, left)
    )
    last = spectra[-1].time_label
    print(f'{label}: {percentages}   so at {last}: "{how_much_is_left(left[-1])}"')

In [ ]:
# The molecule's own peaks against each other in the same file (no acid peak), as a
# percentage of day 1.
print("\nOwn peaks against each other:")
for label, spectra, peak_ppm, other_ppm in (
    ("Ala-Ala alpha at 5.17 against alpha at 4.92", ala_ala_by_date, (5.10, 5.24), (4.82, 5.02)),
    ("Gly-Gly CH2 at 4.77 against CH2 at 4.61    ", gly_gly_by_date, (4.68, 4.86), (4.55, 4.68)),
):
    own = share_of_first(area_ratios(spectra, peak_ppm, other_ppm))
    percentages = ", ".join(
        f"{spectrum.time_label} {share:.0%}" for spectrum, share in zip(spectra, own)
    )
    print(f"{label}: {percentages}")

### Worked example: Gly-Gly

Filled in for you, with the reasons; you fill in Ala-Ala's below.

In [ ]:
GG_WORKED = {
    "call": "survives",  # both CH2 signals are still there, in the same place, at 4 months
    "lost_ppm": [],  # nothing from day 1 is gone
    # a tiny peak that grows from about 8 days on, far smaller than the two main signals
    "new_ppm": [4.33],
    "kept_ppm": [4.77, 4.61],  # the two CH2 signals, one per glycine
    # both CH2 peaks keep a third or more of their day 1 value against the acid peak
    "left": "no clear loss",
    "product_guess": "none",  # nothing new formed in any amount
    "confidence": "high",  # every main signal stayed in place for four months
    "note": (
        "Gly-Gly survives at least 4 months in 98% acid (Table 2 of the paper). One of its two CH2 "
        "peaks, near 4.77 ppm, still weakens: by 4 months it is about a third of its day 1 size "
        "against the acid peak, while the 4.61 ppm peak keeps most of its size and no sizable new "
        "peaks appear. The paper does not explain this, so do not count it as breakdown. From 5 "
        "weeks, the peak near 4.77 ppm shows a small second line beside it; by 4 months the two "
        "lines are equally tall."
    ),
}

### What you need

- **Lost:** a peak at 1 day that is gone, or much smaller, at 4 months.
- **New:** a peak at 4 months that was not there at 1 day. Small new peaks are not your main evidence.
- **Kept:** a peak at both times, in the same place.
- **How much is left:** "little or none" or "no clear loss", as printed under How much is left.

> **RECORD YOUR ANSWER: what changed in Ala-Ala between 1 day and 4 months?**
>
> **What gets saved:** `AA_LOST_PPM`, `AA_NEW_PPM` and `AA_KEPT_PPM`, three lists of peak positions in ppm, and `AA_LEFT`, how much is left.
>
> **Where to look:** the left plot in Part 3 (1 day at the bottom), the lists under it, and the percentages under How much is left.
>
> **How to fill this in:**
> - `AA_LOST_PPM`: peaks there on day 1 and gone or much smaller at 4 months, as numbers like `[1.23, 4.56]`
> - `AA_NEW_PPM`: peaks at 4 months that were not there on day 1
> - `AA_KEPT_PPM`: peaks there at both times
> - `AA_LEFT`: `"little or none"`, `"no clear loss"` or `"unsure"`
> - Leave `[]` if there were none. Gly-Gly's record above shows the format.

<details><summary>Stuck? Open hint 1</summary>

Compare Ala-Ala's two traces one peak at a time: for each day 1 peak, is there still a peak within a few hundredths of a ppm at 4 months?

</details>

<details><summary>Stuck? Open hint 2</summary>

Ala-Ala has four peaks at day 1: two methyls near 2 ppm and two alpha hydrogens near 5 ppm.
The methyl percentage above tells you what happened to the methyl at 1.99 ppm. Then check the alpha
hydrogen at 5.17 ppm by eye.

</details>

In [ ]:
AA_LOST_PPM = []  # peaks there on day 1 and gone or much smaller at 4 months, like [1.23, 4.56]
AA_NEW_PPM = []  # peaks at 4 months that were not there on day 1
AA_KEPT_PPM = []  # peaks there at both times
AA_LEFT = ""  # "little or none", "no clear loss" or "unsure"

# Each line confirms what it saved, next to what the peak picker found.
nt.record.write_down(
    "AA_LOST_PPM", AA_LOST_PPM, "Ala-Ala", "lost_ppm", suggestion=ala_ala_changes["lost"]
)
nt.record.write_down(
    "AA_NEW_PPM", AA_NEW_PPM, "Ala-Ala", "new_ppm", suggestion=ala_ala_changes["new"]
)
nt.record.write_down(
    "AA_KEPT_PPM", AA_KEPT_PPM, "Ala-Ala", "kept_ppm", suggestion=ala_ala_changes["kept"]
)
nt.record.write_down("AA_LEFT", AA_LEFT, "Ala-Ala", "left")

## Part 4: Your decision on Ala-Ala

### What you need

A molecule that survives keeps every peak from hydrogens on carbon in place and at its size relative
to its other peaks, and no new peaks appear. If Ala-Ala breaks down, its fragments can keep reacting
into many small products, which are hard to identify. Base your decision on the large peaks, not on
small new ones. Here the main product's peaks sit almost on top of two of Ala-Ala's, so look for the
peaks that vanish. Your strongest evidence is a methyl peak and an alpha hydrogen peak that
disappear together, because they belong to the same alanine.

If it broke down, decide what is left. Two candidates:
- Free alanine, from splitting the peptide bond with water: your alanine from Part 1, with its
  methyl at 1.98 ppm and its alpha hydrogen at 4.62 ppm in this acid.
- Alaninamide: alanine with an amide group (CONH2, a carbon double-bonded to oxygen and bonded to
  NH2) in place of its carboxyl group (COOH). Its methyl sits near 2.08 ppm and its alpha hydrogen
  near 4.94 ppm. The next cell lays its spectrum, measured in the same acid, over Ala-Ala at 4 months.

Two spectra from separate samples match when the peaks have the same pattern (number of lines and
their spacing) and sit within 0.05 ppm of each other. From 0.05 to 0.1 ppm apart, check the carbon
spectra; more than 0.1 ppm apart is not a match. The same molecule sits 0.01 to 0.07 ppm apart
between samples in this data.

### The second candidate's own spectrum

The next cell lays the mechanism study's spectrum of L-alaninamide, in the same acid, over Ala-Ala at
4 months, with your alanine from Part 1 beside them. Do the remaining peaks match either one, by the
rule above?

In [ ]:
MECHANISM_SPECTRA = "11122291"  # Zenodo record of the mechanism study (Petkowski et al. 2025)
alaninamide_files = nt.data.fetch(MECHANISM_SPECTRA, "L-alaninamide", acid=None)
alaninamide = read_spectrum(alaninamide_files[0])

In [ ]:
labels = ["L-alaninamide (reference)", "alanine, 1 day (Part 1)", "Ala-Ala, 4 months"]
title = "Does the product look like alaninamide?"
traces = [alaninamide, alanine[0], ala_ala_4months]  # bottom to top
fig_reference = plot_stack(traces, READ_PPM, scale_region=READ_PPM, title=title, labels=labels)
plt.show()

In [ ]:
alaninamide_peaks = pick_peaks(alaninamide, READ_PPM, min_height=0.1)
print("alaninamide peaks:", [peak["ppm"] for peak in alaninamide_peaks])

> **RECORD YOUR ANSWER: your decision on Ala-Ala**
>
> **What gets saved:** `AA_CALL`, `AA_PRODUCT`, `AA_CONFIDENCE` and `AA_NOTE`, four short answers in quotes.
>
> **Where to look:** the peaks you saved above, and Gly-Gly's record.
>
> **How to fill this in:**
> - `AA_CALL`: your decision:
>   - `"survives"`: all four peaks are still there at 4 months, in place and at their sizes relative to each other, with no new peaks
>   - `"side chain changed"`: the backbone peaks stay, and new peaks with the same pattern grow beside them
>   - `"breaks down"`: some of the four shrink or vanish while peaks from the products appear and grow
>   - `"unsure"`: the data do not settle it (say why in `AA_NOTE`)
> - `AA_PRODUCT`: the main product in words, `"none"`, or `""` if you have no idea yet
> - `AA_CONFIDENCE`: `"low"`, `"medium"` or `"high"`
> - `AA_NOTE`: anything your reviewer should know, or `""`

<details><summary>Stuck? Open hint 1</summary>

Each alanine has one methyl peak and one alpha hydrogen peak. Did both peaks of one alanine vanish
together? If so, what did the other alanine become?

</details>

<details><summary>Stuck? Open hint 2</summary>

Free alanine has its methyl at 1.98 and its alpha hydrogen at 4.62 ppm in this acid. Do the peaks
that remain at 4 months sit there? If not, the remaining alanine kept a different end: alaninamide.

</details>

In [ ]:
AA_CALL = ""  # your decision: "survives", "side chain changed", "breaks down" or "unsure"
AA_PRODUCT = ""  # the main product, "none", or "" if no idea yet
AA_CONFIDENCE = ""  # "low", "medium" or "high"
AA_NOTE = ""  # anything your reviewer should know

nt.record.write_down("AA_CALL", AA_CALL, "Ala-Ala", "call")
nt.record.write_down("AA_PRODUCT", AA_PRODUCT, "Ala-Ala", "product_guess")
nt.record.write_down("AA_CONFIDENCE", AA_CONFIDENCE, "Ala-Ala", "confidence")
nt.record.write_down("AA_NOTE", AA_NOTE, "Ala-Ala", "note")

### Your verdict

Everything you recorded about Ala-Ala, in one table, next to the Gly-Gly worked example. To change
it, edit a line above and run this cell again.

In [ ]:
MY_VERDICT = nt.record.build_verdict(
    {
        "AA": {
            "call": AA_CALL,
            "lost_ppm": AA_LOST_PPM,
            "new_ppm": AA_NEW_PPM,
            "kept_ppm": AA_KEPT_PPM,
            "left": AA_LEFT,
            "product_guess": AA_PRODUCT,
            "confidence": AA_CONFIDENCE,
            "note": AA_NOTE,
        },
        "GG": GG_WORKED,
    },
    names={"AA": "Ala-Ala", "GG": "Gly-Gly (worked example)"},
    check=["AA"],
    note=AA_NOTE,
)

## Part 5: Compare your decision with the published finding

Table 2 of the paper (Petkowski et al. 2024, *Scientific Reports* 14, 17083) gives Ala-Ala a
stability of 1 day: the last time point at which its spectrum still looks unchanged.
Ala-Ala looks unchanged at 1 day. The change first shows at 3 days, and the loss is gradual: 91% is
left at 3 days, 57% at 8 days, 7% at 5 weeks and none at 4 months. The 3 and 8 day values alone would
be inside the acid peak's drift, but the alpha at 5.17 ppm, measured against the alpha at 4.92 ppm in
the same file, falls in step (85% at 3 days, 69% at 8 days).

The next cell compares your decision with the paper's and your product with the mechanism study
(Petkowski et al. 2025, *ACS Omega* 10, 9623).

The study's mechanism: the end alanine's bond to its nitrogen breaks, so the first alanine leaves as
alaninamide and the end one as acrylic acid, which reacts on into many small products.

In [ ]:
feedback = {}  # by molecule, keyed as in MY_VERDICT: "AA" is Ala-Ala, "GG" is Gly-Gly
if nt.record.revealed("capstone"):
    key = nt.answers.load("capstone")
    for molecule in ("AA", "GG"):
        published = nt.answers.case(key, molecule)
        feedback[molecule] = nt.checks.check_verdict(MY_VERDICT[molecule], published)
        nt.checks.show(feedback[molecule], f"{molecule}: your verdict")

## Hand-in

> **SAVE YOUR HAND-IN: your Ala-Ala hand-in**
>
> **What gets saved:** in `handin/`: the Ala-Ala and Gly-Gly stacks, the 1 day vs 4 months comparison, and `verdict.json` (your verdict table, peak lists and the feedback above).
>
> **Where to look:** nothing new to fill in: the cell uses what you recorded above.
>
> **How to fill this in:**
> - Run the cell. The files are your hand-in for the bootcamp step "Do protein building blocks survive Venus's acid?": the stack is your image, `verdict.json` your note.
> - Nothing is sent for review until you run `nt.portal.submit()` (the last line) or submit on that step.

<details><summary>Stuck? Open hint 1</summary>

If a figure is missing from the list it prints, run the cell that draws it, then this cell again.

</details>

<details><summary>Stuck? Open hint 2</summary>

If it says a name is not defined, run the notebook from the top (Runtime > Run all in Colab).

</details>

In [ ]:
record = nt.record.handin(
    "capstone-verdict",
    figures={"stack_AA": fig_ala_ala, "stack_GG": fig_gly_gly, "zoom_1day_vs_4months": fig_zoom},
    verdict={
        "verdict": MY_VERDICT,
        "reading": MY_READING,
        "my_peaks_ppm": {
            "AA": {"1day": ala_ala_ppm_1day, "4Month": ala_ala_ppm_4months},
            "GG": {"1day": gly_gly_ppm_1day, "4Month": gly_gly_ppm_4months},
        },
        "feedback": feedback,
    },
    numbers={
        "AA_call": MY_VERDICT["AA"]["call"],
        "GG_call": MY_VERDICT["GG"]["call"],
        "AA_product": MY_VERDICT["AA"]["product_guess"],
        "AA_left": MY_VERDICT["AA"]["left"],
        "AA_confidence": MY_VERDICT["AA"]["confidence"],
        "min_height": MIN_HEIGHT,
    },
    question="capstone",
    traps=nt.checks.traps_flagged(traps_ala_ala),
)
# nt.portal.submit()   # when you are ready: sends this step for review